# Deep Learning project

Team members:

* Rimsha Afzal,
* Nika Sharifi Dariani, 265311
* Irina Krylova, 255809

## Setup

Run this notebook from the project root. In Colab, upload or mount the project folder first, then set `PROJECT_ROOT` to that folder.

In [14]:
from pathlib import Path
from typing import Any, Iterable
import json
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Could not find the project root (a folder containing src/). Started from: " + str(Path.cwd()))

sys.path.insert(0, str(PROJECT_ROOT / "src"))

## Paths

The subset query file below is the one compatible with the copied test embeddings. The larger subset query file uses a different index space, so it is not used here.

In [15]:
QUERY_JSON = PROJECT_ROOT / "data/celeba_subset/queries/test_embedding_celeba_evaluation.json"
IMAGE_EMBEDDING_DIR = PROJECT_ROOT / "data/celeba_subset/embeddings/test"
TEXT_EMBEDDING_DIR = PROJECT_ROOT / "data/celeba_subset/embeddings"
CHECK_JSON = PROJECT_ROOT / "data/celeba_subset/checks/query_embedding_compatibility_check.json"
SUMMARY_CSV = PROJECT_ROOT / "outputs/baseline_run/summary.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs/baseline_run/notebook_test_subset_alpha2.00_beta1.00"

for path in [QUERY_JSON, IMAGE_EMBEDDING_DIR, TEXT_EMBEDDING_DIR, CHECK_JSON]:
    if not path.exists():
        raise FileNotFoundError(path)

## Data Sanity Check

Before running retrieval, check that the query file, image embeddings, and text embeddings refer to the same subset.

In [16]:
with QUERY_JSON.open("r", encoding="utf-8") as handle:
    query_entries = json.load(handle)

with CHECK_JSON.open("r", encoding="utf-8") as handle:
    compatibility_check = json.load(handle)

image_embeddings, image_ids = load_embeddings(IMAGE_EMBEDDING_DIR, "image_embeddings", "cpu")
text_embeddings, text_ids = load_embeddings(TEXT_EMBEDDING_DIR, "text_embeddings", "cpu")

query_instances = sum(len(entry["ground_truth"]) for entry in query_entries)
target_links = sum(
    len(targets)
    for entry in query_entries
    for targets in entry["ground_truth"].values()
)

print("compatibility check:", compatibility_check["status"])
print("query types:", len(query_entries))
print("query instances:", query_instances)
print("target links:", target_links)
print("image embeddings:", tuple(image_embeddings.shape), "ids:", len(image_ids))
print("text embeddings:", tuple(text_embeddings.shape), "prompts:", len(text_ids))

compatibility check: pass
query types: 13
query instances: 1640
target links: 26829
image embeddings: (8035, 512) ids: 8035
text embeddings: (40, 512) prompts: 40


### Embeddings & Normalization

Small shared helpers for loading saved embeddings and normalizing vectors.

In [17]:
def l2_normalize(embeddings: torch.Tensor) -> torch.Tensor:
    """Scale one embedding or a batch of embeddings to unit length."""
    single = embeddings.dim() == 1
    if single:
        embeddings = embeddings.unsqueeze(0)
    normalized = F.normalize(embeddings.float(), p=2, dim=1, eps=1e-12)
    return normalized.squeeze(0) if single else normalized


def load_embeddings(output_dir: str | Path, name: str, device: str = "cpu") -> tuple[torch.Tensor, list[Any]]:
    """Load a saved tensor and its matching row ids from disk."""
    output_dir = Path(output_dir)
    embeddings_path = output_dir / f"{name}.pt"
    ids_path = output_dir / f"{name}_ids.npy"

    if not embeddings_path.exists():
        raise FileNotFoundError(f"Missing embeddings file: {embeddings_path}")
    if not ids_path.exists():
        raise FileNotFoundError(f"Missing embedding ids file: {ids_path}")

    try:
        embeddings = torch.load(embeddings_path, map_location=device, weights_only=True)
    except TypeError:
        embeddings = torch.load(embeddings_path, map_location=device)
    ids = np.load(ids_path, allow_pickle=True).tolist()
    return embeddings, ids

# 1. Introduction

# 2. Related work

# 3. Method

[a detailed, formal overview of the solution you developed.
This must include a mathematical description of your architecture, the forward pass, and
the loss functions governing the training process (if applicable). Clearly highlight your
original contributions and adequately cite relevant literature.]

## 3.1. Baseline

The baseline was implemented as a training-free CLIP text-arithmetic retrieval method using the required `openai/clip-vit-base-patch32` model (Radford et al., 2021). The Setup section defines the shared imports, project paths, embedding loader, and L2-normalization helper; this section then contains the baseline-specific query parsing, CLIP arithmetic, retrieval, and metric computation. For each official query, we parse the signed attributes into positive attributes $\mathcal{P}$ and negative attributes $\mathcal{N}$, then construct the composed query vector as $\mathbf{q}=\operatorname{normalize}(\mathbf{v}_{\mathrm{ref}} + \alpha \sum_{p \in \mathcal{P}} \mathbf{t}_p - \beta \sum_{n \in \mathcal{N}} \mathbf{t}_n)$, matching the original `build_query_embeddings(...)` implementation.

Here $\mathbf{v}_{\mathrm{ref}}$ is the reference image embedding, $\mathbf{t}_p$ and $\mathbf{t}_n$ are CLIP text embeddings for the requested CelebA attributes, and $\alpha,\beta$ control the strength of positive and negative edits (baseline sets both values to 1). This follows the CLAY-style idea that semantic edits can be approximated by arithmetic in CLIP space. The composed vector was compared with all gallery image embeddings by dot product, equivalent to cosine similarity after normalization; the reference image itself was excluded from retrieval, and the ranked list was evaluated against the provided CelebA ground-truth targets.

In [20]:
def parse_query_string(query: str) -> tuple[list[str], list[str]]:
    # Step 1: split a signed query such as "+Smiling, -Eyeglasses" into two lists.
    positive = []
    negative = []
    for part in query.split(","):
        part = part.strip()
        if part.startswith("+"):
            positive.append(part[1:].strip())
        elif part.startswith("-"):
            negative.append(part[1:].strip())
    return positive, negative


def attribute_to_prompt(attribute: str) -> str:
    # Step 2: use the same prompt format that was used when saving CLIP text embeddings.
    return f"a photo of a person who is {attribute.replace('_', ' ').lower()}"


def build_query_embeddings(
    reference_embeddings: torch.Tensor,
    positive_embeddings: torch.Tensor | None,
    negative_embeddings: torch.Tensor | None,
    alpha: float = 1.0,
    beta: float = 1.0,
) -> torch.Tensor:
    # Step 3: start from the CLIP image embedding of the reference image.
    queries = reference_embeddings.float()

    # Step 4: add the positive text direction, scaled by alpha.
    if positive_embeddings is not None and positive_embeddings.numel() > 0:
        queries = queries + alpha * positive_embeddings.float().sum(dim=0, keepdim=True)

    # Step 5: subtract the negative text direction, scaled by beta.
    if negative_embeddings is not None and negative_embeddings.numel() > 0:
        queries = queries - beta * negative_embeddings.float().sum(dim=0, keepdim=True)

    # Step 6: normalize so dot product ranking is equivalent to cosine similarity.
    return l2_normalize(queries)


def retrieve_top_k_batch(
    query_embeddings: torch.Tensor,
    image_embeddings: torch.Tensor,
    image_ids: list[int],
    k: int,
    exclude_ids: list[int],
    device: str = "cpu",
) -> list[list[tuple[int, float]]]:
    # Step 7: compute query-to-gallery similarities in CLIP space.
    query_embeddings = query_embeddings.to(device).float()
    image_embeddings = image_embeddings.to(device).float()
    similarities = query_embeddings @ image_embeddings.T

    # Step 8: remove each reference image from its own ranking.
    id_to_row = {int(image_id): row for row, image_id in enumerate(image_ids)}
    for query_row, image_id in enumerate(exclude_ids):
        row = id_to_row.get(int(image_id))
        if row is not None:
            similarities[query_row, row] = -torch.inf

    # Step 9: return the top-k retrieved image ids and similarity scores.
    values, indices = torch.topk(similarities, k=k, dim=1)
    return [
        [(int(image_ids[index]), float(score)) for index, score in zip(row_indices.cpu().tolist(), row_scores.cpu())]
        for row_indices, row_scores in zip(indices, values)
    ]


def recall_at_k(retrieved_indices: list[int], ground_truth_indices: set[int], k: int) -> float:
    # Step 10: Recall@K is 1 if any valid target appears in the top K.
    return 1.0 if set(retrieved_indices[:k]) & ground_truth_indices else 0.0


def precision_at_k(retrieved_indices: list[int], ground_truth_indices: set[int], k: int) -> float:
    # Step 11: Precision@K is the fraction of top-K retrieved images that are valid targets.
    return sum(1 for index in retrieved_indices[:k] if index in ground_truth_indices) / k


def evaluate_single_ranking(
    retrieved_indices: list[int],
    ground_truth_indices: Iterable[int],
    ks: tuple[int, ...] = (1, 5, 10),
) -> dict[str, float]:
    # Step 12: compute all requested retrieval metrics for one reference image.
    ground_truth_set = set(int(index) for index in ground_truth_indices)
    metrics = {}
    for k in ks:
        metrics[f"recall@{k}"] = recall_at_k(retrieved_indices, ground_truth_set, k)
        metrics[f"precision@{k}"] = precision_at_k(retrieved_indices, ground_truth_set, k)
    return metrics


def average_metrics(metric_rows: list[dict[str, float]]) -> dict[str, float]:
    # Step 13: average each metric across all official query/reference instances.
    return {name: sum(row[name] for row in metric_rows) / len(metric_rows) for name in metric_rows[0]}


def load_ground_truth(path: str | Path) -> list[dict[str, Any]]:
    # Step 14: load the professor-provided query and target mapping JSON.
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def run_baseline(
    query_json: str | Path = QUERY_JSON,
    image_embedding_dir: str | Path = IMAGE_EMBEDDING_DIR,
    text_embedding_dir: str | Path = TEXT_EMBEDDING_DIR,
    output_dir: str | Path = BASELINE_OUTPUT_DIR,
    alpha: float = 1.0,
    beta: float = 1.0,
    top_k: tuple[int, ...] = (1, 5, 10),
    batch_size: int = 256,
    device: str = "cpu",
) -> dict[str, Any]:
    # Step 15: create an output folder for this notebook run.
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Step 16: load precomputed CLIP image and text embeddings from the paths defined in Setup.
    image_embeddings, image_ids = load_embeddings(image_embedding_dir, "image_embeddings", "cpu")
    text_embeddings, text_ids = load_embeddings(text_embedding_dir, "text_embeddings", "cpu")

    # Step 17: build lookup tables so JSON ids and prompt strings map to embedding rows.
    image_ids = [int(image_id) for image_id in image_ids]
    image_row = {image_id: row for row, image_id in enumerate(image_ids)}
    text_lookup = {str(prompt): text_embeddings[row] for row, prompt in enumerate(text_ids)}
    entries = load_ground_truth(query_json)

    # Step 18: iterate over each official query and all of its reference images.
    max_k = max(top_k)
    metric_rows = []
    for entry in entries:
        positive, negative = parse_query_string(entry["query"])
        positive_embeddings = torch.stack([text_lookup[attribute_to_prompt(attribute)] for attribute in positive]) if positive else None
        negative_embeddings = torch.stack([text_lookup[attribute_to_prompt(attribute)] for attribute in negative]) if negative else None

        # Step 19: process references in batches to avoid building one huge similarity matrix.
        reference_items = [(int(reference), targets) for reference, targets in entry["ground_truth"].items()]
        for start in range(0, len(reference_items), batch_size):
            batch = reference_items[start:start + batch_size]
            references = [reference for reference, _ in batch]
            reference_embeddings = image_embeddings[[image_row[reference] for reference in references]]

            # Step 20: compose CLIP query embeddings and retrieve top-K gallery images.
            query_embeddings = build_query_embeddings(reference_embeddings, positive_embeddings, negative_embeddings, alpha, beta)
            rankings = retrieve_top_k_batch(query_embeddings, image_embeddings, image_ids, max_k, references, device=device)

            # Step 21: compare every ranking with its valid target set.
            for (_, targets), ranking in zip(batch, rankings):
                retrieved = [image_id for image_id, _ in ranking]
                metric_rows.append(evaluate_single_ranking(retrieved, targets, ks=top_k))

    # Step 22: save and return a single result row for display.
    metrics = average_metrics(metric_rows)
    result = {
        "method": "clip_arithmetic_baseline",
        "alpha": alpha,
        "beta": beta,
        "query_instances": len(metric_rows),
        "gallery_size": len(image_ids),
        **metrics,
    }
    (output_dir / "metrics.json").write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    return result


# Step 23: run the baseline and show the metrics as a DataFrame inside the notebook.
baseline_result = run_baseline(alpha=1.0, beta=1.0, device="cpu")
baseline_results_df = pd.DataFrame([baseline_result])
print(f"Saved metrics to {BASELINE_OUTPUT_DIR / 'metrics.json'}") #TODO: remove later
print(baseline_results_df.to_string(index=False))
baseline_results_df

Saved metrics to /Users/nikadariani/Documents/Courses/Deep Learning/compositional-image-retrieval/outputs/baseline_run/notebook_section_3_1_alpha1.00_beta1.00/metrics.json
                  method  alpha  beta  query_instances  gallery_size  recall@1  precision@1  recall@5  precision@5  recall@10  precision@10
clip_arithmetic_baseline    1.0   1.0             1640          8035      0.05         0.05  0.112195     0.031707   0.170732      0.026098


,method,alpha,beta,query_instances,gallery_size,recall@1,precision@1,recall@5,precision@5,recall@10,precision@10
0,clip_arithmetic_baseline,1.0,1.0,1640,8035,0.05,0.05,0.112195,0.031707,0.170732,0.026098


## 3.2. Baseline sweep / gated fusion grid search

## 3.3. Visual directions

Why it helps? Because text and visual directions live in separate "cones" in the embedding space (SOURCE: CLAY paper, Liang et al. 2022), which is called a modality gap

Liang et al 2022: "They show image and text embeddings in CLIP are "embedded at arm's length" in two completely separate regions of the unit sphere. The Euclidean distance between the image and text embedding clusters for pretrained CLIP is 0.82, and they show this is near the contrastive-loss optimum — in their words, "the default gap distance ‖Δgap‖=0.82 actually achieves the global minimum, and shifting toward closing the gap increases the contrastive loss" — so the gap is preserved by training rather than incidental. The gap originates in the cone effect: at initialization a deep network's outputs occupy a narrow cone. They measure average pairwise cosines of "0.56, 0.47, 0.51 respectively for the 3 models" (ResNet, ViT, Text Transformer), with minimum cosines of only "0.23, 0.05, 0.01" — i.e., embeddings are far from spanning the sphere. This is the root cause of why your text-derived directions and v_ref barely interact in a metrically meaningful way: adding a text vector moves the query partly along the "gap" axis rather than along a semantically discriminative direction inside the image cone."

If we use visual directions, then the signal from the attribute stays in the same cone

## 3.4. Combined text and visual fusion



### 3.4.1. Motivation and formulation

The combined fusion method extends the CLIP text-arithmetic baseline by adding dataset-specific visual attribute directions to the query representation. This design is motivated by CLIP-guided image editing literature. StyleCLIP shows that meaningful semantic edit directions can be derived from CLIP text embeddings, including input-agnostic global directions computed from differences between target and neutral text prompts, and then used to manipulate images through vector operations in a latent space (Patashnik et al., 2021). HairCLIP provides a related precedent for using CLIP to combine multiple conditions, such as hairstyle and hair color specified by text or reference images, in order to perform targeted attribute editing while preserving irrelevant attributes (Wei et al., 2022). Although these works focus on image generation and editing, they support the core idea used here: CLIP embeddings can provide semantic directions, and text and visual conditions can be combined rather than treated as separate retrieval signals.

In our retrieval setting, no generator, mapper, or additional neural network is trained. Instead, we adapt the CLIP-guided editing idea directly in embedding space. For each query, the text direction is computed from the signed positive and negative CLIP attribute embeddings, while the visual direction is computed from CelebA image-embedding statistics for the same attributes. The final hybrid query is

$$
\mathbf{q} = \operatorname{normalize}\left(
\alpha \mathbf{v}_{\mathrm{ref}}
+ \beta_{\mathrm{text}} \Delta_{\mathrm{text}}
+ \beta_{\mathrm{visual,pos}} \Delta_{\mathrm{visual,pos}}
- \beta_{\mathrm{visual,neg}} \Delta_{\mathrm{visual,neg}}
\right),
$$

where $\mathbf{v}_{\mathrm{ref}}$ is the reference image embedding, $\Delta_{\mathrm{text}}$ is the signed CLIP text edit direction, and $\Delta_{\mathrm{visual,pos}}$ and $\Delta_{\mathrm{visual,neg}}$ are sums of CelebA visual attribute directions. The fusion weights control how strongly the query preserves the reference image, follows generic CLIP text semantics, and follows dataset-specific visual attribute evidence. This is the main methodological contribution of the project: the retrieval query combines CLIP's language-guided edit signal with visual directions estimated from the target dataset, then ranks gallery images by cosine similarity after L2 normalization.

In [ ]:
# Add code here

### 3.4.2. Combined text and visual fusion with prompt ensembling

Prompt averaging: Radford et al, 2021 (CLIP paper, they averaged 80 prompts)

Prompt difference: Patashnik et al, 2021. "The "subtract a baseline" construction's actual published precedent is StyleCLIP (Patashnik et al., "StyleCLIP: Text-Driven Manipulation of StyleGAN Imagery," ICCV 2021), the "global directions" method: they build a direction as Δt = normalize(ensemble(target prompts) − ensemble(neutral prompts)) — e.g. "a sports car" minus "a car" — averaged over the CLIP prompt templates."

Prompt ensembling was tested as a refinement of the text component in the combined fusion model. Instead of representing each CelebA attribute with a single prompt, we average multiple prompt templates before inserting the resulting text vector into the hybrid query. This is motivated by CLIP’s original zero-shot classification procedure, where Radford et al. (2021) average predictions across many prompt templates to reduce sensitivity to wording and obtain a more stable text representation. 

We also considered a difference-style prompt construction inspired by StyleCLIP, where Patashnik et al. (2021) compute global edit directions from the difference between target and neutral text embeddings. In our setting, this corresponds to estimating an attribute direction from prompt pairs such as an attribute-specific face prompt minus a neutral face prompt. 

The goal of prompt ensembling is therefore not to change the retrieval architecture, but to make $\Delta_{\mathrm{text}}$ less dependent on a single hand-written phrase and more robust as the text-side edit signal used in the combined text-and-visual query.

In [ ]:
# Add code here

## 3.5. Reliability weighted text and visual fusion combination

The reliability-weighted fusion variant was introduced to address a limitation of using raw visual attribute directions from CelebA. A visual direction estimated as the difference between positive and negative image-embedding means can capture the requested attribute, but it can also capture attributes that are correlated with it in the dataset. This is especially important for CelebA, where facial attributes are not statistically independent. Recent CLIP interpretability work supports this concern: SpLiCE uses CelebA as a case study for detecting and editing spurious concepts in CLIP representations, including the effect of eyewear on identity-related representations (Bhalla et al., 2024). Similarly, Zhao et al. (2025) show that CLIP embedding structure can contain spurious directions on datasets including CelebA, and that identifying or removing such directions can improve robustness. These findings motivate treating visual directions as useful but imperfect evidence rather than as equally reliable attribute edits.

Our reliability-weighted method therefore keeps the same combined text-and-visual fusion form, but scales the contribution of each attribute direction according to a difficulty or reliability score estimated from the dataset. Intuitively, attributes with clean support, lower imbalance, and weaker correlations with unrelated attributes should receive stronger visual-direction weights, while attributes that are rare, highly entangled, or empirically difficult should rely more on the CLIP text direction. This follows the broader principle that language and vision signals can be used together to reduce reliance on spurious visual shortcuts in multimodal models (Yang et al., 2023). The resulting query still uses cosine retrieval after L2 normalization, but the visual part of the edit is no longer treated as uniformly trustworthy across all CelebA attributes.

In [ ]:
# Add code here

### 3.5.1. Correlation-based reliability weighted text and visual fusion combination

The correlation-based variant estimates reliability directly from the CelebA attribute matrix. For each requested attribute, we measure how strongly it co-occurs with the other annotated attributes and use this as a proxy for contamination risk in the visual direction. If an attribute is highly correlated with many other labels, then its mean positive-minus-negative visual direction is more likely to encode a mixture of semantic factors rather than the isolated requested edit. We therefore reduce the visual weight for such attributes and preserve a stronger role for the text direction, which provides a more explicit semantic target. In simplified form, the hybrid query becomes

$$
\mathbf{q} = \operatorname{normalize}\left(
\alpha \mathbf{v}_{\mathrm{ref}}
+ \beta_{\mathrm{text}} \Delta_{\mathrm{text}}
+ \sum_{p \in \mathcal{P}} r_p\,\beta_{\mathrm{visual,pos}}\mathbf{d}_p
- \sum_{n \in \mathcal{N}} r_n\,\beta_{\mathrm{visual,neg}}\mathbf{d}_n
\right),
$$

where $\mathbf{d}_p$ and $\mathbf{d}_n$ are visual attribute directions and $r_a \in [0,1]$ is the reliability score for attribute $a$. Lower reliability means that the attribute direction is more likely to be affected by dataset correlations, so its visual contribution is down-weighted. This mechanism is our correction for the fact that CelebA-derived visual directions are empirical dataset statistics rather than clean causal edits.

In [ ]:
# Add code here

# 4. Experiments and results

Ran experiments on a local subset of N validation and M test images.

For each method, the grid search is run.

[a rigorous description of the training and evaluation strategy. Exten
sively motivate your methodological choices, including network capacity, optimizer selec
tion, hyperparameter tuning, and data sampling strategies]

i suggest to put here also: [an extensive presentation of your findings. You must report stan
dard retrieval metrics (Recall@K). Organize your scores in comparative tables, and in
clude charts depicting learning curves (if applicable), qualitative retrieval examples (suc
cesses and failure cases), and any other visual representations that aid in understanding the
model’s behavior]

## 4.1. Baseline

For each query instance, the baseline builds:

```text
q = normalize(v_ref + alpha * sum(positive_text) - beta * sum(negative_text))
```

where:

- `v_ref` is the CLIP image embedding of the reference image;
- `positive_text` contains CLIP text embeddings for attributes such as `+Smiling`;
- `negative_text` contains CLIP text embeddings for attributes such as `-Eyeglasses`;
- retrieval uses dot product because all embeddings are L2-normalized.

## 4.2. Best Baseline from the Sweep (gated fusion grid search?)

The best setting from the small alpha/beta sweep was `alpha=2.0`, `beta=1.0`.
This cell saves only `metrics.json` to keep experiment folders small.

In [19]:
result = run_baseline(
    query_json=QUERY_JSON,
    image_embedding_dir=IMAGE_EMBEDDING_DIR,
    text_embedding_dir=TEXT_EMBEDDING_DIR,
    output_dir=OUTPUT_DIR,
    alpha=2.0,
    beta=1.0,
    save_predictions=False,
    save_run_config=False,
)

pd.DataFrame([result["metrics"]])

TypeError: run_baseline() got an unexpected keyword argument 'save_predictions'

### Alpha/Beta Sweep Summary

The sweep below compares different text-direction weights. Higher `alpha` means stronger positive attribute push; higher `beta` means stronger negative attribute push.

In [ ]:
summary = pd.read_csv(SUMMARY_CSV)
summary.sort_values("recall@10", ascending=False).reset_index(drop=True)

### Interpretation

The baseline is intentionally simple: it does not learn a transformation and does not use visual attribute directions.
The best run in the current sweep, `alpha=2.0` and `beta=1.0`, improves recall@10 over the default `alpha=1.0`, `beta=1.0`.

This suggests that, on this subset, a stronger positive text direction helps. The next natural improvement is to replace raw text directions with visual attribute directions, or to add a small gating mechanism that controls how strongly each attribute modifies the reference image.

These numbers are useful for development, but they are subset results. Final assignment numbers should be reported on the official full CelebA test split when the full test embeddings are available.

## 4.3. Visual-Direction Retrieval

After the CLIP text-arithmetic baseline, we tested a second retrieval variant based on visual attribute directions. For each CelebA attribute, the direction was computed from the train split as:

```text
direction(attribute) = normalize(mean(images where attribute = +1) - mean(images where attribute = -1))
```

The retrieval query then used the reference image plus signed visual directions:

```text
q = normalize(alpha * reference_image + beta_pos * sum(positive directions) - beta_neg * sum(negative directions))
```

This was implemented separately from the baseline in `src/retrieval/visual_direction_retrieval.py`, so the baseline results remain a fixed reference point.


### Visual-Direction Sweep

We evaluated a small fixed-weight grid using the same test subset, query file, and metrics as the baseline:

```text
alpha:    0.5, 1.0, 2.0
beta_pos: 0.5, 1.0, 2.0
beta_neg: 0.5, 1.0, 2.0
```

The sweep was run with `scripts/run_visual_direction_retrieval.py`, and the sorted results were saved to `outputs/visual_direction_run/summary.csv`.


In [ ]:
visual_summary_path = PROJECT_ROOT / "outputs/visual_direction_run/summary.csv"
visual_summary = pd.read_csv(visual_summary_path)
visual_summary.head(5)[[
    "alpha", "beta_pos", "beta_neg",
    "recall@1", "precision@1",
    "recall@5", "precision@5",
    "recall@10", "precision@10",
]]

### Baseline vs Visual Directions

The best visual-direction setting was compared against the best baseline setting. The visual-direction run was close, but it did not improve over the CLIP text-arithmetic baseline.


In [ ]:
best_baseline = summary.sort_values("recall@10", ascending=False).iloc[0]
best_visual = visual_summary.sort_values("recall@10", ascending=False).iloc[0]

comparison = pd.DataFrame([
    {
        "method": "CLIP text arithmetic baseline",
        "alpha": best_baseline["alpha"],
        "beta": best_baseline["beta"],
        "beta_pos": None,
        "beta_neg": None,
        "recall@1": best_baseline["recall@1"],
        "recall@5": best_baseline["recall@5"],
        "recall@10": best_baseline["recall@10"],
        "precision@10": best_baseline["precision@10"],
    },
    {
        "method": "visual directions",
        "alpha": best_visual["alpha"],
        "beta": None,
        "beta_pos": best_visual["beta_pos"],
        "beta_neg": best_visual["beta_neg"],
        "recall@1": best_visual["recall@1"],
        "recall@5": best_visual["recall@5"],
        "recall@10": best_visual["recall@10"],
        "precision@10": best_visual["precision@10"],
    },
])
comparison


### Visual-Direction Takeaway

The best visual-direction setting was `alpha=2.0`, `beta_pos=1.0`, `beta_neg=0.5`, with `recall@1=0.0506`, `recall@5=0.1463`, and `recall@10=0.2183`. The best baseline setting was `alpha=2.0`, `beta=1.0`, with `recall@1=0.0561`, `recall@5=0.1500`, and `recall@10=0.2262`. Therefore, simple fixed visual directions were competitive but slightly worse than text directions. This suggests that raw visual directions alone may be noisy or too global, and the next improvement should probably combine text and visual directions or weight attributes based on difficulty/reliability. The strongest visual-direction runs consistently favored a smaller negative-direction weight, with the best setting using `beta_neg=0.5`; this suggests that negative visual directions are less trustworthy in this setup and may be more destructive than positive directions when they are applied too strongly.


## 4.4. Combined Text + Visual Fusion

Since visual directions alone may be noisy and may contain correlated CelebA attribute information, we tested a query-level hybrid that keeps the CLIP text direction as the main semantic signal and adds visual directions as a dataset-specific correction. The query is:

```text
q = normalize(
    alpha * reference_image_embedding
    + beta_text * text_delta
    + beta_visual_pos * visual_pos_delta
    - beta_visual_neg * visual_neg_delta
)
```

where `text_delta = sum(text_embeddings[pos_attrs]) - sum(text_embeddings[neg_attrs])`, `visual_pos_delta = sum(visual_directions[pos_attrs])`, and `visual_neg_delta = sum(visual_directions[neg_attrs])`. This was implemented in `src/retrieval/hybrid_fusion_retrieval.py` and run with `scripts/run_hybrid_fusion_retrieval.py`.


In [ ]:
hybrid_summary_path = PROJECT_ROOT / "outputs/hybrid_fusion_run/summary.csv"
hybrid_summary = pd.read_csv(hybrid_summary_path)
best_hybrid = hybrid_summary.sort_values("recall@10", ascending=False).iloc[0]

hybrid_comparison = pd.DataFrame([
    {
        "method": "CLIP text arithmetic baseline",
        "alpha": best_baseline["alpha"],
        "beta_text": best_baseline["beta"],
        "beta_visual_pos": None,
        "beta_visual_neg": None,
        "recall@1": best_baseline["recall@1"],
        "recall@5": best_baseline["recall@5"],
        "recall@10": best_baseline["recall@10"],
        "precision@10": best_baseline["precision@10"],
    },
    {
        "method": "visual directions",
        "alpha": best_visual["alpha"],
        "beta_text": None,
        "beta_visual_pos": best_visual["beta_pos"],
        "beta_visual_neg": best_visual["beta_neg"],
        "recall@1": best_visual["recall@1"],
        "recall@5": best_visual["recall@5"],
        "recall@10": best_visual["recall@10"],
        "precision@10": best_visual["precision@10"],
    },
    {
        "method": "hybrid text + visual fusion",
        "alpha": best_hybrid["alpha"],
        "beta_text": best_hybrid["beta_text"],
        "beta_visual_pos": best_hybrid["beta_visual_pos"],
        "beta_visual_neg": best_hybrid["beta_visual_neg"],
        "recall@1": best_hybrid["recall@1"],
        "recall@5": best_hybrid["recall@5"],
        "recall@10": best_hybrid["recall@10"],
        "precision@10": best_hybrid["precision@10"],
    },
])

hybrid_comparison


### Combination Takeaway

The best hybrid setting was `alpha=1.0`, `beta_text=2.0`, `beta_visual_pos=0.5`, and `beta_visual_neg=0.25`, with `recall@1=0.0671`, `recall@5=0.1927`, `recall@10=0.2860`, and `precision@10=0.0413`. This improves over both the best text-only baseline (`recall@10=0.2262`) and the best visual-direction-only run (`recall@10=0.2183`). The smaller negative visual weight again supports the hypothesis that negative visual directions are less reliable than positive ones when used too strongly.

This parameter pattern is also informative. The text-only baseline needed a larger reference weight (`alpha=2.0`), while the hybrid works best with a smaller reference weight (`alpha=1.0`) and a stronger modification signal (`beta_text=2.0`). This suggests that, once the attribute modification is supported by both CLIP text embeddings and CelebA visual directions, the query can move farther away from the original reference image and rely more on the requested edit.

A natural next step is therefore a finer grid around this region:

```text
alpha:           0.5, 0.75, 1.0, 1.25, 1.5
beta_text:       1.5, 2.0, 2.5, 3.0
beta_visual_pos: 0.25, 0.5, 0.75, 1.0
beta_visual_neg: 0.0, 0.1, 0.25, 0.4, 0.5
```

The `beta_visual_neg=0.0` case is especially important: if negative visual directions are very noisy, the best hybrid may use visual directions only for positive attributes and leave negative edits to the CLIP text direction.



### 4.3.1. Prompt-Ensemble Hybrid Check

As a small follow-up, we replaced the single CLIP attribute prompt in the hybrid method with an average of multiple positive prompt templates per attribute. The retrieval formula stays the same; only the text embedding used inside `text_delta` changes.


In [ ]:
prompt_ensemble_summary_path = PROJECT_ROOT / "outputs/hybrid_prompt_ensemble_run/summary.csv"
prompt_ensemble_summary = pd.read_csv(prompt_ensemble_summary_path)
best_prompt_ensemble = prompt_ensemble_summary.sort_values("recall@10", ascending=False).iloc[0]

pd.DataFrame([
    {
        "method": "single-prompt hybrid",
        "recall@10": best_hybrid["recall@10"],
        "precision@10": best_hybrid["precision@10"],
    },
    {
        "method": "prompt-ensemble hybrid",
        "recall@10": best_prompt_ensemble["recall@10"],
        "precision@10": best_prompt_ensemble["precision@10"],
    },
])


### Prompt-Ensemble Takeaway

Prompt ensembling did not improve the main retrieval score in this experiment: the single-prompt hybrid reached `recall@10=0.2860`, while the prompt-ensemble hybrid reached `recall@10=0.2811`. This suggests that averaging several text prompts mostly smooths the text signal, but does not add a useful correction beyond what the single prompt and CelebA visual directions already provide.

The prompt-ensemble run has a slightly higher `precision@10` (`0.0416` vs `0.0413`), but the difference is very small, so it is better treated as a side observation rather than the main direction. The strongest method remains the hybrid with the original CLIP text prompt plus visual directions: CLIP text provides the semantic edit, and the CelebA visual directions provide the dataset-specific adjustment.


## 4.5. Reliability weighted hybrid text and visual fusion


### 4.5.1. Correlation-based reliability weighted hybrid text and visual fusion

# 5. Discussion and conclusion

# 6. References

Liang, V. W., Zhang, Y., Kwon, Y., Yeung, S., & Zou, J. Y. (2022). Mind the gap: Understanding the modality gap in multi-modal contrastive representation learning. Advances in Neural Information Processing Systems, 35, 17612-17625.

Radford, A., Kim, J. W., Hallacy, C., Ramesh, A., Goh, G., Agarwal, S., ... & Sutskever, I. (2021, July). Learning transferable visual models from natural language supervision. In International conference on machine learning (pp. 8748-8763). PmLR.

Patashnik, O., Wu, Z., Shechtman, E., Cohen-Or, D., & Lischinski, D. (2021). Styleclip: Text-driven manipulation of stylegan imagery. In Proceedings of the IEEE/CVF international conference on computer vision (pp. 2085-2094).

Wei, T., Chen, D., Zhou, W., Liao, J., Tan, Z., Yuan, L., Zhang, W., & Yu, N. (2022). HairCLIP: Design your hair by text and reference image. In Proceedings of the IEEE/CVF Conference on Computer Vision and Pattern Recognition (pp. 18072-18081).

Bhalla, U., Oesterling, A., Srinivas, S., Calmon, F. P., & Lakkaraju, H. (2024). Interpreting CLIP with sparse linear concept embeddings. arXiv preprint arXiv:2402.10376.

Yang, Y., Nushi, B., Palangi, H., & Mirzasoleiman, B. (2023). Mitigating spurious correlations in multi-modal models during fine-tuning. arXiv preprint arXiv:2304.03916.

Zhao, J., Li, C., Sala, F., & Rohe, K. (2025). Quantifying structure in CLIP embeddings: A statistical framework for concept interpretation. arXiv preprint arXiv:2506.13831.